# 05 — End-to-End Pipeline Demo

Runs the full two-stage cascade pipeline and measures real-world performance.

```
Frame → Stage 1 Detector → crops
                            ├─ Pothole  → PotholeAnalyzer  → severity, distance, instructions
                            └─ Traffic  → TrafficAnalyzer  → color, lane_relevant, instructions
```

**Sections:**
1. Pipeline load + GPU profile
2. Inference speed benchmark (FPS)
3. Single image full breakdown
4. Pothole CV score radar charts
5. Traffic light color detection internals
6. Batch grid (9 images)
7. Video inference (optional)

In [ ]:
import sys, time
sys.path.insert(0, '../src')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
import torch

BASE_DIR = Path('..').resolve()
VAL_DIR  = BASE_DIR / 'data' / 'processed' / 'detector_yolo' / 'images' / 'val'

from pipeline import Pipeline

pipe = Pipeline()
print('Pipeline ready.')

## 1. Inference Speed Benchmark

Warm up the GPU then measure FPS over 30 frames.

In [ ]:
val_imgs = sorted(VAL_DIR.glob('*'))[:50]
assert val_imgs, f'No val images found in {VAL_DIR} — run prepare_data.py first'

frames = [cv2.imread(str(p)) for p in val_imgs[:30] if cv2.imread(str(p)) is not None]

# GPU warm-up (2 frames, not counted)
for f in frames[:2]:
    pipe.run(f)
if torch.cuda.is_available():
    torch.cuda.synchronize()

# Benchmark
t0 = time.time()
for f in frames:
    pipe.run(f)
if torch.cuda.is_available():
    torch.cuda.synchronize()
elapsed = time.time() - t0

fps = len(frames) / elapsed
ms  = elapsed / len(frames) * 1000

print(f'Frames tested   : {len(frames)}')
print(f'Total time      : {elapsed:.3f}s')
print(f'Per frame       : {ms:.1f} ms')
print(f'FPS             : {fps:.1f}')
print()
if fps >= 25:
    print('Real-time capable (≥ 25 FPS)')
elif fps >= 15:
    print('Near real-time (15–25 FPS) — suitable for dashcam with minor buffering')
else:
    print(f'Below real-time ({fps:.1f} FPS) — consider reducing conf threshold or using YOLOv8n')

## 2. Single Image — Full Analysis Breakdown

In [ ]:
img_path = frames[0] if isinstance(frames[0], np.ndarray) else cv2.imread(str(val_imgs[0]))
frame    = img_path if isinstance(img_path, np.ndarray) else cv2.imread(str(img_path))
result   = pipe.run(frame)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(cv2.cvtColor(frame,                    cv2.COLOR_BGR2RGB))
axes[0].set_title('Original Frame', fontsize=12)
axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(result['annotated_frame'], cv2.COLOR_BGR2RGB))
axes[1].set_title('Annotated — Stage 1 + Stage 2', fontsize=12)
axes[1].axis('off')
plt.tight_layout()
plt.show()

print('=== WARNINGS ===')
for w in result['warnings'] or ['No warnings — no lane-relevant detections.']:
    print(' ', w)

## 3. Pothole Analysis Detail

In [ ]:
if result['potholes']:
    print(f'{len(result["potholes"])} pothole(s) detected:\n')
    for i, ph in enumerate(result['potholes']):
        print(f'  #{i+1}  conf={ph["conf"]:.0%}  severity={ph["severity"]}  '
              f'distance={ph["distance"]}  area={ph["area_px"]}px')
        print(f'       {ph["instructions"]}')
        if ph.get('cv_scores'):
            s = ph['cv_scores']
            print(f'       CV → edge={s["edge_density"]:.3f}  depth={s["depth_score"]:.3f}  '
                  f'texture={s["texture_score"]:.3f}  combined={s["combined"]:.3f}')
        print()
else:
    print('No potholes in this frame.')

## 4. Pothole CV Score Radar Charts

In [ ]:
# Collect potholes from multiple frames for richer visualisation
all_potholes = []
for f in frames[:20]:
    r = pipe.run(f)
    all_potholes.extend(r['potholes'])

ph_with_scores = [p for p in all_potholes if p.get('cv_scores')][:6]

if ph_with_scores:
    keys   = ['edge_density', 'depth_score', 'texture_score']
    labels = ['Edge\nDensity', 'Depth\nScore', 'Texture\nScore']
    angles = np.linspace(0, 2*np.pi, len(keys), endpoint=False).tolist() + [0]

    sev_clr = {'Low': '#27ae60', 'Medium': '#f39c12', 'High': '#e74c3c'}
    n = len(ph_with_scores)
    fig, axes = plt.subplots(1, n, figsize=(4*n, 4), subplot_kw={'polar': True})
    if n == 1: axes = [axes]

    for ax, ph in zip(axes, ph_with_scores):
        vals  = [ph['cv_scores'][k] for k in keys]
        vals_n = [min(v / 0.3, 1.0) for v in vals] + [min(vals[0]/0.3, 1.0)]
        clr   = sev_clr.get(ph['severity'], 'gray')
        ax.plot(angles, vals_n, color=clr, linewidth=2)
        ax.fill(angles, vals_n, color=clr, alpha=0.2)
        ax.set_thetagrids(np.degrees(angles[:-1]), labels, fontsize=8)
        ax.set_ylim(0, 1)
        ax.set_title(f'{ph["severity"]}\n{ph["conf"]:.0%}', color=clr, fontsize=10, pad=12)

    plt.suptitle('Pothole CV Scores (normalised, cap=0.3)', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print('No pothole CV scores found in first 20 frames.')

## 5. Traffic Light Color Detection Internals

In [ ]:
from analyzers.traffic_analyzer import _detect_color

tl_crops = []
for f in frames[:40]:
    preds = pipe.detector(f, conf=0.40, verbose=False)
    for box in preds[0].boxes:
        if int(box.cls[0]) == 1:
            x1,y1,x2,y2 = [int(v) for v in box.xyxy[0]]
            crop = f[y1:y2, x1:x2]
            if crop.size > 0:
                tl_crops.append(crop)

if tl_crops:
    n = min(len(tl_crops), 5)
    c_map = {'Red': '#e74c3c', 'Yellow': '#f1c40f', 'Green': '#2ecc71', 'Unknown': 'gray'}
    fig, axes = plt.subplots(2, n, figsize=(3*n, 6))

    for col, crop in enumerate(tl_crops[:n]):
        color, pos_vote, color_vote, conf = _detect_color(crop)

        axes[0][col].imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        axes[0][col].axis('off')
        axes[0][col].set_title('Crop', fontsize=8)

        resized = cv2.resize(crop, (48, 144))
        axes[1][col].imshow(cv2.cvtColor(resized, cv2.COLOR_BGR2RGB), aspect='auto')
        axes[1][col].axhline(48,  color='white', linewidth=1.5, linestyle='--')
        axes[1][col].axhline(96,  color='white', linewidth=1.5, linestyle='--')
        axes[1][col].set_xticks([]); axes[1][col].set_yticks([])
        tc = c_map.get(color, 'white')
        axes[1][col].set_title(
            f'{color}\nPos:{pos_vote}\nHSV:{color_vote}\nConf:{conf:.1f}',
            fontsize=7, color=tc
        )

    plt.suptitle('Traffic Light Color Detection\n(dashed = spatial thirds)', fontsize=11)
    plt.tight_layout()
    plt.show()
else:
    print('No traffic light crops found in the first 40 frames.')

## 6. Batch Grid — 9 Validation Images

In [ ]:
grid_frames = [cv2.imread(str(p)) for p in val_imgs[:9]]
fig, axes   = plt.subplots(3, 3, figsize=(15, 10))

for ax, f in zip(axes.flatten(), grid_frames):
    if f is None: continue
    r   = pipe.run(f)
    ann = cv2.cvtColor(r['annotated_frame'], cv2.COLOR_BGR2RGB)
    ax.imshow(ann)
    ax.axis('off')
    ph_n = len(r['potholes'])
    tl_n = sum(1 for t in r['traffic_lights'] if t['lane_relevant'])
    sev  = r['potholes'][0]['severity'] if r['potholes'] else ''
    clr  = r['traffic_lights'][0]['color'] if r['traffic_lights'] else ''
    ax.set_title(f'PH:{ph_n}{" "+sev if sev else ""}  TL:{tl_n}{" "+clr if clr else ""}',
                 fontsize=8)

plt.suptitle('Batch Inference — Validation Set', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Traffic Light Detail

In [ ]:
if result['traffic_lights']:
    print(f'{len(result["traffic_lights"])} traffic light(s) detected:\n')
    for i, tl in enumerate(result['traffic_lights']):
        lane = 'YOUR LANE' if tl['lane_relevant'] else 'adjacent lane'
        print(f'  #{i+1}  conf={tl["conf"]:.0%}  color={tl["color"]}  {lane}')
        print(f'       pos_vote={tl.get("position_vote")}  '
              f'hsv_vote={tl.get("color_vote")}  '
              f'confidence={tl.get("color_confidence",0):.2f}')
        print(f'       {tl["instructions"]}')
        print()
else:
    print('No traffic lights in this frame.')

## 8. Video Inference (Optional)

Set `VIDEO_PATH` to a dashcam `.mp4` file.

In [ ]:
VIDEO_PATH  = None   # e.g. '../sample_dashcam.mp4'
OUTPUT_PATH = '../output_annotated.mp4'

if VIDEO_PATH and Path(VIDEO_PATH).exists():
    t0     = time.time()
    result = pipe.run_video(VIDEO_PATH, OUTPUT_PATH)
    elapsed = time.time() - t0
    n_frames = result['frame_count']
    print(f'Processed {n_frames} frames in {elapsed:.1f}s ({n_frames/elapsed:.1f} FPS)')
    print(f'Output → {OUTPUT_PATH}')
else:
    print('Set VIDEO_PATH to a dashcam video file to run.')